In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

<b><font size='3'> 문제에서 요구하는 DataFrame을 보일 때, 최상위 10개의 행만 보이시오.</font></b>

<b><font size='3'> 정답을 찾지 못하더라도, 정답을 찾는 중간 과정의 DataFrame을 보이고, 설명을 추가할 경우, 부분 점수가 부여됩니다.</font></b>

<font size='7'> <b>1.</b></font><font size='5'><span style="line-height:1.7"> <b>netflix_titles.csv</b>, <b>netflix_titles.csv</b>, <b>netflix_titles.csv</b>을 DataFrame으로 로드하고, <font color='blue'>Schema</font>를 확인하시오.</span></font>

<font size='4'> <b>dataset 1. netflix_titles.csv</b> </font>
<font size='3'>
> <font color='blue'><b>id</b></font>: netflix 작품의 id </br>
> <font color='blue'><b>title</b></font>: 작품의 제목</br>
> <font color='blue'><b>type</b></font>: 작품의 종류 (MOVIE 또는 SHOW)</br>
> <font color='blue'><b>release_year</b></font>: 공개 년도</br>
> <font color='blue'><b>age_certification</b></font>: 관람 등급(G,NC-17,PG,PG-13,R,TV-14,TV-G,TV-MA,TV-PG,TV-Y,TV-Y7)</br>
> <font color='blue'><b>runtime</b></font>: 러닝타임</br>
> <font color='blue'><b>genres</b></font>: 장르 (action, animation, comedy, crime, documentation, drama, family, fantasy, horror, music, reality, romance, scifi, thriller, war, western, 혼합 장르)</br>
> <font color='blue'><b>production_contries</b></font>: 제작 국가</br>
</font>

<font size='4'> <b>dataset 2. netflix_imdb_rating.csv</b> </font>
<font size='3'>
> <font color='blue'><b>id</b></font>: netflix 작품의 id </br>
> <font color='blue'><b>type</b></font>: 작품의 종류 (MOVIE 또는 SHOW)</br>
> <font color='blue'><b>imdb_id</b></font>: imdb 작품의 id</br>
> <font color='blue'><b>imdb_score</b></font>: imdb 평점</br>
> <font color='blue'><b>imdb_votes</b></font>: imdb 평가 수</br>
</font>

<font size='4'> <b>dataset 3. netflix_credits.csv</b> </font>
<font size='3'>
> <font color='blue'><b>person_id</b></font>: 감독 또는 출연자의 id </br>
> <font color='blue'><b>id</b></font>: netflix 작품의 id</br>
> <font color='blue'><b>name</b></font>: 감독 또는 출연자의 이름</br>
> <font color='blue'><b>character</b></font>: 캐릭터</br>
> <font color='blue'><b>role</b></font>: 역할 (ACTOR 또는 DIRECTOR)</br>
</font>

In [3]:
spark = SparkSession.builder.appName('02').config('spark.driver.host','localhost').getOrCreate()

In [5]:
netflix_title = spark.read.csv('netflix_titles.csv',header=True,inferSchema=True)
netflix_title.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)



In [7]:
netflix_rating = spark.read.csv('netflix_imdb_ratings.csv',header=True,inferSchema=True)
netflix_rating.printSchema()

root
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: integer (nullable = true)



In [12]:
netflix_credit = spark.read.csv('netflix_credits.csv',header=True,inferSchema=True)
netflix_credit.printSchema()

root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)



<font size='7'> <b>2.</b></font><font size='5'><span style="line-height:1.7"> <font color='blue'>모든 감독 또는 출연자</font>에 대해 <b>감독 또는 출연자의 이름</b>, 출연 또는 감독한 각 <b>작품의 제목</b>, 각 작품의 <b>imdb 평점</b>을 보이시오.</span></font>
> <font size='3'> 감독 또는 출연자의 이름, 작품의 제목은 중복될 수 있다.</font>

In [43]:
netflix = netflix_credit.join(netflix_rating,['id'],how='left')
netflix = netflix.join(netflix_title,['id'],how='left')

In [44]:
netflix.select('name','title','imdb_score').show(10)

+---------------+-----------+----------+
|           name|      title|imdb_score|
+---------------+-----------+----------+
| Robert De Niro|Taxi Driver|       8.3|
|   Jodie Foster|Taxi Driver|       8.3|
|  Albert Brooks|Taxi Driver|       8.3|
|  Harvey Keitel|Taxi Driver|       8.3|
|Cybill Shepherd|Taxi Driver|       8.3|
|    Peter Boyle|Taxi Driver|       8.3|
| Leonard Harris|Taxi Driver|       8.3|
| Diahnne Abbott|Taxi Driver|       8.3|
|    Gino Ardito|Taxi Driver|       8.3|
|Martin Scorsese|Taxi Driver|       8.3|
+---------------+-----------+----------+
only showing top 10 rows



<font size='7'> <b>3.</b></font><font size='5'><span style="line-height:1.7"> <font color='blue'>감독(DIRECTOR)을 제외한</font> <b>출연자(ACTOR)의 이름</b>, 출연한 <b>작품의 제목</b>과 <font color='blue'>소문자로 변환</font>한 <b>캐릭터</b> 문자열을 보이되, <font color='blue'>작품의 제목과 캐릭터는 하나의 column</font>으로 보이시오.</span></font></br>
 > <span style="line-height:2.0"><font size='3'>- 작품의 제목과 캐릭터를 <b>대시(-)로 연결</b>하며, column명은 <b>title_and_character</b>로 함 </br>
 > ex> <b>title</b>이 <font color='blue'>Taxi Driver</font>, <b>character</b>가 <font color='blue'>Travis Bickle</font>인 경우, 새로운 열(<b>title_and_character</b>)의 값을 <font color='blue'>Taxi Driver - TRAVIS BICKLE</font>로 변환함</font></span>

In [66]:
net_ac = netflix.where(netflix.role.isin('ACTOR'))
net_ac = net_ac.select('name','title','character',lower(net_ac.character).alias('lowCH'))
net_ac = net_ac.select('name','title','lowCH',concat_ws(' - ',net_ac.title,net_ac.lowCH).alias('title_and_character'))
net_ac.select('name','title_and_character').show(10)

+---------------+--------------------+
|           name| title_and_character|
+---------------+--------------------+
| Robert De Niro|Taxi Driver - tra...|
|   Jodie Foster|Taxi Driver - iri...|
|  Albert Brooks|   Taxi Driver - tom|
|  Harvey Keitel|Taxi Driver - mat...|
|Cybill Shepherd| Taxi Driver - betsy|
|    Peter Boyle|Taxi Driver - wizard|
| Leonard Harris|Taxi Driver - sen...|
| Diahnne Abbott|Taxi Driver - con...|
|    Gino Ardito|Taxi Driver - pol...|
|Martin Scorsese|Taxi Driver - pas...|
+---------------+--------------------+
only showing top 10 rows



<font size='7'> <b>4.</b></font><font size='5'><span style="line-height:1.7"> <font color='blue'><b>imdb 평가 수</b>가 100000 이상</font>인 작품은 <b>yes</b>, 나머지 작품는 <b>no</b> 값을 갖는 <b>popularity</b> column을 생성하고, <b>작품의 제목, imdb 평점, imdb 평가 수</b> column을 <font color='blue'><b>imdb 평점</b> 값을 기준으로 내림차순 정렬</font>하시오.</span></font>

In [45]:
net_im = netflix.withColumn('popularity',when(netflix.imdb_votes > 100000,'yes').otherwise('no'))
net_im.select('title','imdb_score','imdb_votes','popularity').orderBy(net_im['imdb_score'].desc()).show(10)

+------------+----------+----------+----------+
|       title|imdb_score|imdb_votes|popularity|
+------------+----------+----------+----------+
|        NULL|     195.0|      NULL|        no|
|        NULL|     195.0|      NULL|        no|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
|Breaking Bad|       9.5|   1727694|       yes|
+------------+----------+----------+----------+
only showing top 10 rows



<font size='7'> <b>5.</b></font><font size='5'><span style="line-height:1.7"> <b><font color='blue'>관람 등급</font></b>이 <font color='blue'>PG</font> 또는 <font color='blue'>PG-13</font>인 작품 중 <font color='blue'><b>imdb 평점</b></font>이 <font color='blue'>7.0 이상</font>인 <b>작품의 제목, imdb 평점, 장르, 관람 등급</b>을 보이시오.</span></font>

In [68]:
net_pg = netflix.select('title','imdb_score','genres','age_certification').where(netflix.age_certification.isin('PG','PG-13'))
net_pg.where(net_pg.imdb_score>7.0).show(10)

+--------------------+----------+--------------------+-----------------+
|               title|imdb_score|              genres|age_certification|
+--------------------+----------+--------------------+-----------------+
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
|Monty Python and ...|       8.2|['fantasy', 'acti...|               PG|
+--------------------+----------+------------------

<font size='7'> <b>6.</b></font><font size='5'><span style="line-height:1.7"> <font color='blue'>감독(DIRECTOR)을 제외한 <b>10편 이상</b>의 작품에 출연한 <b>출연자(ACTOR)</b></font>들 중 <font color='blue'><b>imdb 평점 평균이 가장 높은 배우 3명의 이름</b></font>을 찾으시오.</span></font>
> <font size='3'> <b>Hint: </b>10편 이상 출연한 출연자는, 각 출연 배우의 이름을 기준으로 그룹화하고, 집계값을 영화의 id 또는 title의 count로 하여 확인할 수 있다.

In [65]:
from pyspark.sql.functions import *
net_hi = netflix.where(netflix.role.isin('ACTOR'))
net_hi = net_hi.select('name','imdb_score','title').groupBy(net_hi.name).agg(count(net_hi.title).alias('count_actor'),mean('imdb_score').alias('mean_score'))
net_hi = net_hi.where(net_hi['count_actor']>=10).orderBy(net_hi['mean_score'].desc())
net_hi.select('name').show(3)

+--------------+
|          name|
+--------------+
|Graham Chapman|
|   Terry Jones|
|     Eric Idle|
+--------------+
only showing top 3 rows

